<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">از کاراکتر تا نمایش یادگرفتنی</h1><p style="text-align:right"><b>پرسش آزمایش:</b> شناسه، سطر <bdi dir="ltr">Embedding</bdi> و موقعیت چه فرق‌هایی دارند؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-04/chapter-01/21-tokenizer.html"><bdi dir="ltr">21-tokenizer</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-04/chapter-02/23-shift.html"><bdi dir="ltr">23-shift</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/25-embedding.html"><bdi dir="ltr">25-embedding</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/26-positions.html"><bdi dir="ltr">26-positions</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">از <bdi dir="ltr">Tokenizer</bdi> و <bdi dir="ltr">Dataset</bdi> واقعی پروژه استفاده می‌کنیم. جدول <bdi dir="ltr">Embedding</bdi> این آزمایش کوچک و تصادفی است؛ از آن دربارهٔ معنای واژه‌ها نتیجه نمی‌گیریم. <bdi dir="ltr">Vocabulary</bdi> فقط از متن مرجعِ تعیین‌شده ساخته می‌شود.</p>
</div>

In [ ]:
from torch import nn
from mini_gpt.tokenizer import CharacterTokenizer
from mini_gpt.dataset import NextTokenDataset
reference = "مدل می‌رود. مدل می‌آید."
tokenizer = CharacterTokenizer.from_text(reference)
print(list(enumerate(tokenizer.id_to_token)))
ids = tokenizer.encode(reference)
dataset = NextTokenDataset(ids, context_length=6)
x, y = dataset[0]
print("input:", x.tolist(), tokenizer.decode(x.tolist()))
print("target:", y.tolist(), tokenizer.decode(y.tolist()))
assert torch.equal(x[1:], y[:-1])
inspect("token IDs", x)
print("Unknown example:", tokenizer.encode("🐈"), tokenizer.decode(tokenizer.encode("🐈")))


<div dir="rtl" style="text-align:right">
<p style="text-align:right">پیش‌بینی کنید دو بار آمدن یک شناسه، پیش از افزودن بردار موقعیت چه خروجی می‌دهد. آیا بزرگ‌تر بودن شناسه به معنی بزرگ‌تر بودن ویژگی‌هاست؟</p>
</div>

In [ ]:
C = 4
embedding = nn.Embedding(tokenizer.vocab_size, C)
position = nn.Embedding(6, C)
same_ids = torch.tensor([[1,1,2]], dtype=torch.long)
tokens = embedding(same_ids)                       # (B,T,C)
positions = position(torch.arange(same_ids.shape[1]))  # (T,C)
combined = tokens + positions
for name, value in [("IDs",same_ids),("Token Embedding",tokens),
                    ("Position Embedding",positions),("Combined",combined)]:
    inspect(name, value)
    print(value.detach())
torch.testing.assert_close(tokens[0,0], tokens[0,1])
assert not torch.equal(combined[0,0], combined[0,1])
one_hot = torch.nn.functional.one_hot(same_ids, tokenizer.vocab_size).float()
torch.testing.assert_close(one_hot @ embedding.weight, tokens)


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">کدام عددها <bdi dir="ltr">Gradient</bdi> می‌گیرند؟</h2><p style="text-align:right">شناسه‌ها عدد صحیح و نشانی سطرند؛ <bdi dir="ltr">Parameter</bdi>های جدول یاد می‌گیرند. <bdi dir="ltr">Loss</bdi> زیر فقط مجموع عددهاست تا مسیر مشتق دیده شود، نه هدف آموزش زبان. حدس بزنید چرا سطر شناسهٔ ۱ دو برابر سهم می‌گیرد.</p>
</div>

In [ ]:
embedding.zero_grad(set_to_none=True)
embedding(same_ids).sum().backward()
inspect("Embedding Gradient", embedding.weight.grad)
print(embedding.weight.grad)
torch.testing.assert_close(embedding.weight.grad[1], torch.full((C,),2.))
torch.testing.assert_close(embedding.weight.grad[2], torch.ones(C))
assert same_ids.grad is None
try:
    embedding(torch.tensor([tokenizer.vocab_size]))
except (IndexError, RuntimeError) as error:
    print("Expected out-of-vocabulary index:", error)
else:
    raise AssertionError("Expected invalid index")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> <bdi dir="ltr">Tokenizer</bdi> فعلی را ثابت نگه دارید. نسخهٔ تازه‌ای از متن بسازید و در آن «ی» را با «ي» یا فاصله را با نیم‌فاصله عوض کنید؛ همان <bdi dir="ltr">Tokenizer</bdi> را روی ورودی تازه اجرا کنید. انتظار دارید طول، شناسه‌ها و نرخ ناشناخته چگونه تغییر کند؟ سپس <bdi dir="ltr">context_length</bdi> را تغییر دهید و زوج ورودی/هدف تازه را دستی بررسی کنید. جمع بردار <bdi dir="ltr">Token</bdi> و بردار موقعیت هنوز اطلاعات موقعیت‌های مختلف را با هم ترکیب نمی‌کند؛ این ترکیب را در <bdi dir="ltr">Attention</bdi> بررسی می‌کنیم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/26-positions.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: تکرار شناسه، جمع‌شدن سهم یک سطر</h2>
<p style="text-align:right">از تعداد تکرار <bdi dir="ltr">Token</bdi>ها، <bdi dir="ltr">Gradient</bdi> یک <bdi dir="ltr">Embedding</bdi> ساده را پیش‌بینی کنید. پیش‌نیاز: <bdi dir="ltr">Tokenizer</bdi> و <bdi dir="ltr">Lookup</bdi> و آزمایش <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">sum</code> در همین دفتر را دیده‌اید؛ این <bdi dir="ltr">Loss</bdi> هدف آموزش زبان نیست. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر شناسهٔ ۱ سه بار و شناسهٔ ۲ یک بار دیده شود، مشتق <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">sum</code> خروجی نسبت به هر ویژگیِ این دو سطر چیست؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch import nn
from mini_gpt.tokenizer import CharacterTokenizer
tokenizer = CharacterTokenizer.from_text('سلام مدل')
ids = torch.tensor([[1,1,2],[3,1,2]])
print('IDs:',ids,'vocabulary size:',tokenizer.vocab_size)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">lookup_gradient_counts(ids,vocab_size,channels)</code> مشتقِ پیش‌بینی‌شدهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">embedding(ids).sum()</code> را به شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(V,C)</code> برگرداند. بدون <bdi dir="ltr">Autograd</bdi>، تعداد حضور هر <bdi dir="ltr">ID</bdi> را در همهٔ ویژگی‌های همان سطر بگذارید.</p>
</div>

In [ ]:
def lookup_gradient_counts(ids, vocab_size, channels):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = lookup_gradient_counts(ids,tokenizer.vocab_size,4)
    if result is None: return False
    for tokens,C in ((ids,4),(torch.tensor([[0,2,0,2]]),3)):
        table = nn.Embedding(tokenizer.vocab_size,C)
        table(tokens).sum().backward()
        torch.testing.assert_close(lookup_gradient_counts(tokens,tokenizer.vocab_size,C),table.weight.grad)
    assert torch.equal(result[1],torch.full((4,),3.))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط یک <bdi dir="ltr">ID</bdi> دیگر با مقدار ۱ اضافه کنید. برای این <bdi dir="ltr">Loss</bdi> خاص، همهٔ ویژگی‌های همان سطر یک واحد سهم بیشتر می‌گیرند؛ این قانون عمومی هر <bdi dir="ltr">Loss</bdi> زبانی نیست.</p>
</div>

In [ ]:
before = torch.bincount(ids.flatten(),minlength=tokenizer.vocab_size)
after = torch.bincount(torch.cat((ids.flatten(),torch.tensor([1]))),minlength=tokenizer.vocab_size)
print('row count change:',after-before)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب <bdi dir="ltr">ID</bdi> صفر را از شمارش حذف می‌کند چون آن را «نبود داده» می‌پندارد. در <bdi dir="ltr">Tokenizer</bdi> پروژه، صفر سطر واقعی <bdi dir="ltr">Unknown</bdi> است. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">count_all_ids(ids,vocab_size)</code> همهٔ شناسه‌ها را بشمارد.</p>
</div>

In [ ]:
unknown_ids = torch.tensor([0,1,0,2])
print('wrong: omitted unknown occurrences:',torch.bincount(unknown_ids[unknown_ids!=0],minlength=tokenizer.vocab_size))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def count_all_ids(ids, vocab_size):
    # TODO
    return None

In [ ]:
def test_repair():
    result = count_all_ids(unknown_ids,tokenizer.vocab_size)
    if result is None: return False
    assert result[0].item() == 2
    assert result.sum().item() == 4
    assert count_all_ids(torch.tensor([[2,2,1]]),4).tolist() == [0,1,2,0]
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CharacterTokenizer</code> واقعی، <bdi dir="ltr">Unknown</bdi> را با <bdi dir="ltr">ID</bdi> صفر نگه می‌دارد. <bdi dir="ltr">Token Embedding</bdi> این سطر را مانند سطرهای دیگر می‌خواند؛ شناسه‌ها <bdi dir="ltr">Gradient</bdi> ندارند، اما سطرهای خوانده‌شده می‌توانند داشته باشند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام ویژگیِ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">sum</code> باعث شد <bdi dir="ltr">Gradient</bdi> هر سطر فقط به تعداد تکرار آن وابسته باشد؟ در <bdi dir="ltr">Loss</bdi> واقعی چه چیزی علاوه بر تعداد تکرار اثر دارد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/26-positions.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-06_tokens_embeddings.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>